# Laya V2 — Fine-Tuning on `LocalLLaMA/typed-decisions` (Kaggle GPU T4 ×2 DDP)

[![PyPI version](https://img.shields.io/pypi/v/laya.svg)](https://pypi.org/project/laya/)
[![Hugging Face Model](https://img.shields.io/badge/%F0%9F%A4%97%20Model-convaiinnovations%2Flaya-blue)](https://huggingface.co/convaiinnovations/laya)
[![Dataset](https://img.shields.io/badge/%F0%9F%A4%97%20Dataset-LocalLLaMA%2Ftyped--decisions-green)](https://huggingface.co/datasets/LocalLLaMA/typed-decisions)

This notebook fine-tunes **Laya** (`convaiinnovations/laya`, 421M params) on the **1,200 training cases (6,000 typed decisions)** of the [LocalLLaMA/typed-decisions](https://huggingface.co/datasets/LocalLLaMA/typed-decisions) benchmark using **both NVIDIA T4 GPUs via Distributed Data Parallel (`torchrun --nproc_per_node=2`)**.

---

### ⚠️ Kaggle Notebook Settings:
In the right-hand sidebar panel under **Notebook options**:
* **Accelerator:** Select **`GPU T4 x2`** (ensure both GPUs are allocated)
* **Internet:** Set to **`On`**
* **Output:** Checkpoints and final model are saved to `/kaggle/working/laya_finetuned_typed_decisions`

---
### V2 changes (vs V1: Acc 0.764 / Soft 0.534 / ECE 0.223)
1. **Calibration fitted on typed-decisions only** (case-level 10% hold-out). V1 fitted temperatures on a random mix dominated by hard-label extra data, which mis-scaled confidence (ECE 0.223).
2. **Extra data no longer swamps the benchmark**: capped at 400/source (V1: 1,500) and down-weighted (0.3) in the loss.
3. **Label smoothing (0.10)** on hard-label extras so they cannot teach one-hot overconfidence (hurt soft accuracy / ECE).
4. **LR warm-up + correct cosine length** (V1 scheduler could climb back up after `T_max`), 5 epochs.
5. New customer-service source (Bitext); every dataset block is isolated so one bad schema is skipped, never crashes the run.

---
**V3:** CE-only loss, typed-decisions-only data, per-(qtype, option-bucket) temperatures, per-question noul thresholds, 10 epochs (LR head 5e-5 / encoder 1e-5), DDP `find_unused_parameters=False`. Run cell 6 again (items now carry `qid`).


---
**V4**: V2 recipe restored (extras on, LR 2.5e-5/1e-4, 6 epochs), 6% hold-out, best-epoch selection, calibration fitted on hard-label top-1 correctness (V3's soft-NLL fit worsened ECE), per-question noul thresholds off.


## Verified notebook notes
This file has been statically checked for Python syntax in ordinary code cells. Full GPU training and Section 6 evaluation must still be run in Kaggle with Internet and GPU T4 x2 enabled. Section 4 writes the training script; Section 5 performs the time-consuming training. Section 6 deliberately stops if neither a final nor recoverable checkpoint exists.


## Important run order (corrected)

1. Set **Internet = On** and **Accelerator = GPU T4 x2** in Kaggle Notebook options.
2. Run the GPU check cell.
3. Run the pinned installation cell once, then run the compatibility-check cell immediately after it.
4. Continue in order only if the compatibility check passes. If it fails, stop and keep the full error output; do not run the training/evaluation cells.
5. Training can take a long time. Do not use **Run All** until the environment check passes.

This file pins package versions to prevent the open-ended upgrade that installed Transformers 5.x. Static checks can validate notebook syntax and references, but the full GPU training run still depends on Kaggle's live package availability, internet access, and allocated GPUs.


## 1. Environment & Dual T4 GPU Check
Verify both T4 GPUs are detected.


In [ ]:
!nvidia-smi
import os, subprocess, torch

n_gpu = torch.cuda.device_count()
print(f"CUDA Available: {torch.cuda.is_available()} | Visible GPUs: {n_gpu}")
for i in range(n_gpu):
    p = torch.cuda.get_device_properties(i)
    print(f"  GPU {i}: {p.name} ({p.total_memory / 1e9:.1f} GB)")

assert n_gpu >= 2, (
    f"Expected 2 GPUs, but detected {n_gpu}!\n"
    "Please switch your Kaggle Accelerator: on the right sidebar, go to Notebook options -> "
    "Accelerator -> select GPU T4 x2."
)

os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"
print("Both T4 GPUs verified and ready for DDP training!")


## 2. Install Dependencies


In [ ]:
# Install a pinned environment matching this notebook's legacy laya.agent/laya.common API.
# Do not use -U with open-ended version ranges here: that pulled Transformers 5.x and broke imports.
%pip install --no-cache-dir "laya==0.1.6" "transformers==4.48.3" "tokenizers==0.21.0" "datasets==3.3.2" "huggingface_hub==0.29.3" "safetensors==0.5.3" "accelerate==1.4.0" "pyarrow==19.0.1" "pandas==2.2.3" "scipy==1.15.2" "tabulate==0.9.0"
print("Install cell finished. Run the next verification cell before continuing.")


In [ ]:
import sys, importlib.metadata as md
print("Python:", sys.version)
for package in ["laya", "transformers", "tokenizers", "datasets", "huggingface-hub", "torch"]:
    try:
        print(f"{package}: {md.version(package)}")
    except md.PackageNotFoundError:
        print(f"{package}: NOT INSTALLED")

# These imports are the compatibility gate. Do not run training if this cell fails.
import transformers
from laya.agent import _fix_tokenizer_config
from laya.common import build_model, build_sequence, render_options, QTYPES
print("Compatibility check passed: expected legacy Laya API is available.")


## 3. Download & Preprocess Data for DDP
We preprocess all 1,200 training cases into tokenized items and save them to disk so both DDP worker ranks can read them.


In [ ]:
import os, json, torch, sys, importlib.util
from datasets import load_dataset
from transformers import AutoTokenizer
from huggingface_hub import snapshot_download
try:
    from laya.agent import _fix_tokenizer_config
    from laya.common import build_sequence, render_options, QTYPES
except ModuleNotFoundError as e:
    raise ModuleNotFoundError(
        "The `laya` package is unavailable in this Kaggle session. "
        "Run Section 2 (Install Dependencies) again, confirm it finishes, "
        "then rerun this cell. If the install cell just ran, restart the session "
        "and run the notebook from Section 2 onward."
    ) from e

MODEL_ID = "convaiinnovations/laya"
print(f"Fetching tokenizer and config from {MODEL_ID}...")
model_dir = snapshot_download(MODEL_ID)
_fix_tokenizer_config(model_dir)

tok = AutoTokenizer.from_pretrained(os.path.join(model_dir, "tokenizer"))
with open(os.path.join(model_dir, "rl_agent_config.json")) as f:
    cfg = json.load(f)

print("Downloading LocalLLaMA/typed-decisions (train split)...")
ds_train = load_dataset("LocalLLaMA/typed-decisions", "all", split="train")

def build_training_item(state, q, gold_q):
    t = q["type"]
    crit = q.get("criteria", {})
    if t == "choice":
        keys = list(crit.keys())
        target = [gold_q["probabilities"].get(k, 0.0) for k in keys]
    elif t == "noul":
        target = [gold_q["probabilities"].get("false", 0.5), gold_q["probabilities"].get("true", 0.5)]
    elif t == "score":
        n_levels = len(crit) if isinstance(crit, list) else 4
        target = [gold_q["probabilities"].get(str(i), 0.0) for i in range(n_levels)]
    
    s = sum(target)
    target = [v / s for v in target] if s > 0 else [1.0 / len(target)] * len(target)
    label = target.index(max(target))
    k = len(render_options({"t": t, "crit": crit}))
    
    seq, markers = build_sequence(tok, state, {"t": t, "ins": q["instructions"], "crit": crit}, cfg["max_len"], cfg["head_max_len"])
    if len(markers) != k:
        return None
    return {
        "ids": seq,
        "markers": markers,
        "qtype": QTYPES[t],
        "target": target,
        "label": label
    }

items = []
for cid, row in enumerate(ds_train):
    state = json.loads(row["state"])
    questions = json.loads(row["questions"])
    gold = json.loads(row["gold"])
    for qid, q in questions.items():
        if qid in gold:
            it = build_training_item(state, q, gold[qid])
            if it:
                it["src"] = "td"; it["cid"] = cid; it["qid"] = qid
                items.append(it)

print(f"Preprocessed {len(items)} training sequences across 1,200 cases.")
torch.save(items, "/kaggle/working/train_items.pt")
print("Saved preprocessed items to /kaggle/working/train_items.pt")


## 3b. Add Extra Datasets (V2)
Same Typed-Decisions conversion as V1, but: **400 rows/source**, **label-smoothed targets**, each item tagged with its source (`src`) so training can down-weight extras and calibration can use typed-decisions items only. A source that fails to load or has a different schema is skipped with a printed message.


In [ ]:
import re, random
from collections import Counter

MAX_PER_DATASET = 400    # V2: cap per source (V1 used 1500 and swamped the benchmark data)
LABEL_SMOOTH = 0.10      # V2: hard labels -> smoothed targets (prevents one-hot overconfidence)
rng = random.Random(20260930)
n_td = len(items)

def _txt(*parts):
    return re.sub(r"\s+", " ", " ".join(str(p) for p in parts if p)).strip()[:2000]

def _load(*specs):
    for name, kw in specs:
        try:
            ds = load_dataset(name, split="train", **kw)
            print(f"  loaded {name}: {len(ds)} rows")
            return ds
        except Exception as e:
            print(f"  could not load {name}: {type(e).__name__}: {str(e)[:100]}")
    return None

def _pick(cols, options):
    return next((c for c in options if c in cols), None)

def _lab(ds, col, v):
    f = ds.features[col]
    return f.names[v] if hasattr(f, "names") else str(v)

def _smooth_choice(keys, gold):
    K = len(keys); e = LABEL_SMOOTH if K > 1 else 0.0
    return {k: (1 - e) if k == gold else e / max(K - 1, 1) for k in keys}

def _smooth_ordinal(n, g):
    p = [0.0] * n
    if n == 1:
        p[0] = 1.0; return p
    e = LABEL_SMOOTH
    nb = [j for j in (g - 1, g + 1) if 0 <= j < n]
    p[g] = 1 - e
    for j in nb: p[j] += e / len(nb)
    return p

def qc(ins, keys, gold):
    keys = list(dict.fromkeys(keys))
    assert gold in keys, f"gold {gold!r} not in options"
    return ({"type": "choice", "instructions": ins, "criteria": {k: "" for k in keys}},
            {"probabilities": _smooth_choice(keys, gold), "label": gold})

def qs(ins, levels, gold_idx):
    levels = list(levels); p = _smooth_ordinal(len(levels), gold_idx)
    return ({"type": "score", "instructions": ins, "criteria": levels},
            {"probabilities": {str(i): p[i] for i in range(len(levels))},
             "label": gold_idx, "score": float(gold_idx)})

RANK = {"informational": 0, "info": 0, "very low": 0, "very_low": 0, "low": 1, "minor": 1,
        "medium": 2, "moderate": 2, "high": 3, "major": 3, "critical": 4, "severe": 4}

def _levels(values):
    return sorted({v for v in values if v in RANK}, key=lambda v: (RANK[v], v))

def _sample(ds):
    return [ds[i] for i in rng.sample(range(len(ds)), min(MAX_PER_DATASET, len(ds)))]

def add_cases(name, cases):
    n = 0
    for state, questions, gold in cases:
        for qid, q in questions.items():
            it = build_training_item(state, q, gold[qid])
            if it:
                it["src"] = name; it["cid"] = -1
                items.append(it); n += 1
    print(f"  {name}: +{n} training sequences")

def safe(name, fn):
    print(name)
    try:
        fn()
    except Exception as e:
        print(f"  SKIPPED {name}: {type(e).__name__}: {str(e)[:150]}")

def intent_cases(rows, text_fn, label_fn, names, ins, channel, field, qid):
    cases = []
    for r in rows:
        g = label_fn(r); t = text_fn(r)
        if not t or g not in names: continue
        pool = [n for n in names if n != g]
        cand = rng.sample(pool, min(7, len(pool))) + [g]; rng.shuffle(cand)
        q, a = qc(ins, cand, g)
        cases.append(({"channel": channel, field: t}, {qid: q}, {qid: a}))
    return cases

def d_banking():
    ds = _load(("PolyAI/banking77", {}), ("PolyAI/banking77", {"revision": "refs/convert/parquet"}),
               ("legacy-datasets/banking77", {}))
    if ds is None: return
    names = [n.replace("_", " ") for n in ds.features["label"].names]
    add_cases("banking77", intent_cases(_sample(ds), lambda r: _txt(r["text"]), lambda r: names[r["label"]],
              names, "Which banking intent does the customer message express?",
              "banking_customer_message", "message", "banking_intent"))

def d_itsupport():
    ds = _load(("tasksource/it-support-tickets", {}))
    if ds is None: return
    names = [str(n) for n in ds.features["label"].names]
    cases = []
    for r in _sample(ds):
        q, a = qc("Which IT support category does this ticket belong to?", names, names[r["label"]])
        cases.append(({"channel": "it_support_ticket", "ticket": _txt(r["text"])},
                      {"it_support_category": q}, {"it_support_category": a}))
    add_cases("it-support-tickets", cases)

def d_tickets():
    ds = _load(("Tobi-Bueck/customer-support-tickets", {}))
    if ds is None: return
    rows = _sample(ds)
    queues = [k for k, _ in Counter(str(r["queue"]).strip() for r in rows if r.get("queue")).most_common(10)]
    prios = _levels(str(r["priority"]).strip().lower() for r in rows if r.get("priority"))
    cases = []
    for r in rows:
        text = _txt(r.get("subject"), r.get("body"))
        qs_, gs_ = {}, {}
        if text and str(r.get("queue")).strip() in queues:
            qs_["ticket_queue"], gs_["ticket_queue"] = qc("Which support queue should this ticket be routed to?", queues, str(r["queue"]).strip())
        p = str(r.get("priority")).strip().lower()
        if text and p in prios and len(prios) >= 2:
            qs_["ticket_priority"], gs_["ticket_priority"] = qs("How urgent is this ticket?", prios, prios.index(p))
        if qs_:
            cases.append(({"channel": "customer_support_ticket", "ticket": text}, qs_, gs_))
    print("  queues:", queues, "| priority levels:", prios)
    add_cases("customer-support-tickets", cases)

def d_email():
    ds = _load(("elenigkove/Email_Intent_Classification", {}))
    if ds is None: return
    tc, lc = _pick(ds.column_names, ["Email", "email", "text", "body"]), _pick(ds.column_names, ["Intent", "intent", "label"])
    if not tc or not lc: print("  columns not found:", ds.column_names); return
    lab = lambda r: r[lc] if isinstance(r[lc], str) else _lab(ds, lc, r[lc])
    rows = _sample(ds)
    labs = sorted({lab(r) for r in ds})
    cases = []
    for r in rows:
        q, a = qc("What is the intent of this email?", labs, lab(r))
        cases.append(({"channel": "email", "email": _txt(r[tc])}, {"email_intent": q}, {"email_intent": a}))
    add_cases("email-intent", cases)

def d_incidents():
    ds = _load(("emmanuelgjr/genai-incidents", {}))
    if ds is None: return
    rows = _sample(ds)
    sev = lambda r: str(r.get("severity")).strip().lower()
    levels = _levels(sev(r) for r in rows)
    print("  severity levels:", levels)
    if len(levels) < 2: return
    cases = []
    for r in rows:
        text = _txt(r.get("title"), r.get("description"))
        if text and sev(r) in levels:
            q, a = qs("How severe is this security incident?", levels, levels.index(sev(r)))
            cases.append(({"channel": "security_incident", "incident": text}, {"incident_severity": q}, {"incident_severity": a}))
    add_cases("genai-incidents", cases)

def d_bitext():   # NEW in V2: customer-service intent (matches the customer_service workflow)
    ds = _load(("bitext/Bitext-customer-support-llm-chatbot-training-dataset", {}))
    if ds is None: return
    tc, lc = _pick(ds.column_names, ["instruction", "text"]), _pick(ds.column_names, ["intent", "category"])
    if not tc or not lc: print("  columns not found:", ds.column_names); return
    names = sorted({str(v).replace("_", " ") for v in ds.unique(lc)})
    add_cases("bitext-customer-service", intent_cases(_sample(ds), lambda r: _txt(r[tc]),
              lambda r: str(r[lc]).replace("_", " "), names, "Which customer-service intent does this message express?",
              "customer_service_message", "message", "cs_intent"))

for nm, fn in [("BANKING77", d_banking), ("IT Support", d_itsupport), ("Customer Support Tickets", d_tickets),
               ("Email Intent", d_email), ("Security Incidents", d_incidents), ("Bitext Customer Service", d_bitext)]:
    safe(nm, fn)

print(f"\nTyped-decisions items: {n_td} | new-dataset items: {len(items) - n_td} | total: {len(items)}")
print("Per-source counts:", dict(Counter(it.get("src") for it in items)))
assert n_td > 0, "No typed-decisions items were built - check cell 3"
torch.save(items, "/kaggle/working/train_items.pt")
print("Saved combined items to /kaggle/working/train_items.pt")


## 4. DDP Training Script (`train_ddp.py`)
We write the multi-GPU distributed RLCD training script using pure policy gradients with proper scoring rules and DDP gradient synchronization.


### Experiment 1: Learning-rate adjustment
The training script below uses a lower encoder/head learning rate than the original V4 baseline. Keep `EXTRA_W = 0.3` unchanged for this run so the effect of learning rate can be measured separately. Compare accuracy, soft accuracy, Brier score, and ECE against the saved V4 baseline. This is an experiment, not a guaranteed improvement. Keep a copy of the original V4 metrics before running.

**Experiment 2 (run separately after recording Experiment 1):** change only `EXTRA_W` from `0.3` to `0.2` in the training script, retrain, and compare the same metrics. Do not change both settings in the same run if you want to know which change helped. Restore the original learning rates for this second experiment so each result isolates one variable.

**Reproducibility note:** use the same official test split for every run; do not tune settings against test examples. Record each run's metrics and settings separately.

In [ ]:
%%writefile /kaggle/working/train_ddp.py
import os, sys, time, json, random, math
import numpy as np
import torch
import torch.distributed as dist
from torch.nn.parallel import DistributedDataParallel as DDP
from safetensors.torch import load_file, save_file
from transformers import AutoTokenizer
from laya.common import build_model, QTYPES
try:
    from laya.common import temp_bucket, clamp_temperature
except ImportError:   # older laya: same definitions as the runtime lookup
    def temp_bucket(qtype, k):
        size = "2" if k <= 2 else "3-5" if k <= 5 else "6-10" if k <= 10 else "11+"
        return "%s:%s" % (["choice", "score", "noul"][int(qtype)], size)
    def clamp_temperature(t, lo=0.5, hi=5.0):
        try: return float(min(max(float(t), lo), hi))
        except Exception: return 1.0

TEMP_MIN, TEMP_MAX = 0.5, 5.0     # the runtime clamps to this range, so fit inside it
MIN_TYPE_N, MIN_BUCKET_N = 10, 30  # hold-out is ~600 items, so the runtime's 2000 floor is unreachable

def collate_train_batch(items, pad_id):
    n, L = len(items), max(len(it["ids"]) for it in items)
    kmax = max(len(it["markers"]) for it in items)
    ids = torch.full((n, L), pad_id, dtype=torch.long)
    att = torch.zeros((n, L), dtype=torch.long)
    mpos = torch.zeros((n, kmax), dtype=torch.long)
    mmask = torch.zeros((n, kmax), dtype=torch.bool)
    target = torch.zeros((n, kmax), dtype=torch.float32)
    for i, it in enumerate(items):
        ids[i, : len(it["ids"])] = torch.tensor(it["ids"])
        att[i, : len(it["ids"])] = 1
        k = len(it["markers"])
        mpos[i, :k] = torch.tensor(it["markers"])
        mmask[i, :k] = True
        target[i, : len(it["target"])] = torch.tensor(it["target"], dtype=torch.float32)
    return {
        "input_ids": ids,
        "attention_mask": att,
        "marker_pos": mpos,
        "marker_mask": mmask,
        "target": target,
        "qtype": torch.tensor([it["qtype"] for it in items]),
        "label": torch.tensor([it["label"] for it in items]),
        "weight": torch.tensor([float(it.get("w", 1.0)) for it in items], dtype=torch.float32)
    }

def fit_one_temp(sel, min_n=MIN_TYPE_N):
    """V4: fit T so top-1 confidence matches top-1 CORRECTNESS vs the hard gold label (binary cross-entropy,
    a proper score). V3 minimised NLL against the flat soft teacher targets, which pushes T>1 and made
    hold-out ECE worse (0.112 -> 0.148). Grid search inside the runtime's [0.5, 5.0] clamp."""
    if len(sel) < min_n:
        return 1.0
    best_t, best_l = 1.0, float("inf")
    for T in np.round(np.arange(TEMP_MIN, TEMP_MAX + 1e-9, 0.05), 2):
        l = 0.0
        for z, t in sel:
            p = softmax_np(z, T); c = float(np.clip(p.max(), 1e-6, 1 - 1e-6))
            y = float(np.argmax(p) == int(np.argmax(t)))
            l -= y * np.log(c) + (1 - y) * np.log(1 - c)
        if l < best_l:
            best_l, best_t = l, float(T)
    return clamp_temperature(best_t, TEMP_MIN, TEMP_MAX)

def softmax_np(z, T=1.0):
    z = np.asarray(z, dtype=np.float64) / T
    e = np.exp(z - z.max())
    return e / e.sum()

def ece_np(confs, corrects, n_bins=15):
    confs, corrects = np.asarray(confs, float), np.asarray(corrects, float)
    if len(confs) == 0:
        return float("nan")
    edges = np.linspace(0, 1, n_bins + 1); e = 0.0
    for lo, hi in zip(edges[:-1], edges[1:]):
        m = (confs > lo) & (confs <= hi)
        if m.any():
            e += m.mean() * abs(confs[m].mean() - corrects[m].mean())
    return float(e)

def macro_f1(p_true, y, thr):
    pred = p_true >= thr; f = []
    for cls in (True, False):
        tp = np.sum((pred == cls) & (y == cls)); fp = np.sum((pred == cls) & (y != cls)); fn = np.sum((pred != cls) & (y == cls))
        f.append(0.0 if tp == 0 else 2 * tp / (2 * tp + fp + fn))
    return float(np.mean(f))

def fit_noul_threshold(p_true, y, min_n=20, min_gain=0.01):
    """Threshold on P(true) maximising macro-F1 (F1 of true and of false averaged, so predicting
    everything 'true' cannot win). Falls back to 0.5 when data is thin or the gain is negligible."""
    p_true, y = np.asarray(p_true, float), np.asarray(y, bool)
    if len(y) < min_n or y.sum() < 5 or (~y).sum() < 5:
        return 0.5
    grid = np.round(np.arange(0.20, 0.801, 0.01), 2)
    scores = np.array([macro_f1(p_true, y, g) for g in grid])
    best = grid[np.lexsort((np.abs(grid - 0.5), -scores))[0]]   # best score, ties -> closest to 0.5
    return float(best) if scores.max() - macro_f1(p_true, y, 0.5) >= min_gain else 0.5

def eval_holdout(model, items, tok, device):
    """Soft-CE and argmax accuracy on held-out typed-decisions items (rank 0 only, no collectives)."""
    model.eval(); nll = acc = n = 0.0
    with torch.no_grad():
        for i in range(0, len(items), 16):
            ch = items[i:i + 16]; cb = collate_train_batch(ch, tok.pad_token_id)
            with torch.autocast("cuda", dtype=torch.float16):
                l, _ = model(cb["input_ids"].to(device), cb["attention_mask"].to(device),
                             cb["marker_pos"].to(device), cb["marker_mask"].to(device), cb["qtype"].to(device))
            m = cb["marker_mask"].to(device); t = cb["target"].to(device)
            lp = torch.log_softmax(l.float().masked_fill(~m, -1e4), -1)
            nll += (-(t * lp).sum(-1)).sum().item(); acc += (lp.argmax(-1) == t.argmax(-1)).sum().item(); n += len(ch)
    model.train()
    return nll / max(n, 1), acc / max(n, 1)

def main():
    dist.init_process_group("nccl")
    rank = dist.get_rank()
    world_size = dist.get_world_size()
    local_rank = int(os.environ.get("LOCAL_RANK", "0"))
    torch.cuda.set_device(local_rank)
    device = torch.device("cuda", local_rank)

    model_dir = sys.argv[1]
    output_dir = sys.argv[2]
    
    with open(os.path.join(model_dir, "rl_agent_config.json")) as f:
        cfg = json.load(f)
    cfg["gradient_checkpointing"] = True
    cfg["max_tokens_per_batch"] = 4096
    cfg["max_len"] = 1024
    cfg["head_max_len"] = 256

    tok = AutoTokenizer.from_pretrained(os.path.join(model_dir, "tokenizer"))
    model = build_model(cfg, encoder_dir=os.path.join(model_dir, "encoder"))
    
    weights = load_file(os.path.join(model_dir, "model.safetensors"))
    model.load_state_dict(weights, strict=True)
    
    model.encoder.gradient_checkpointing_enable(gradient_checkpointing_kwargs={"use_reentrant": False})
    model.head_checkpointing = True
    model.to(device)
    model.train()

    all_params = [p for p in model.parameters() if p.requires_grad]
    ddp_model = DDP(model, device_ids=[local_rank], find_unused_parameters=False)
    
    all_items = torch.load("/kaggle/working/train_items.pt", weights_only=False)
    # V3: extras carry hard labels that do not match the soft benchmark targets -> train on
    # typed-decisions only. Set True to mix them back in (still down-weighted, no soft-logit step).
    USE_EXTRAS = True   # V4: extras helped (V2 0.782 / customer_service 0.758 -> 0.726 when removed)
    if not USE_EXTRAS:
        all_items = [it for it in all_items if it.get("src", "td") == "td"]

    # Hold the calibration slice out of training before sharding. Temperatures fitted on
    # items the run has already trained on measure the fit rather than the calibration: the
    # model is near-certain and near-correct on them, so the optimiser has nothing to soften
    # and returns a degenerate scale. The seed is fixed and rank-independent, so every rank
    # withholds exactly the same items and none of them reaches a training batch.
    # V2: hold out whole typed-decisions CASES for calibration (never trained on). Temperatures are
    # fitted on benchmark-distribution items only, with soft gold targets, so extra hard-label data
    # cannot distort the confidence scale. Seed is fixed -> identical on every rank.
    EXTRA_W = 0.3        # loss weight for non-benchmark (extra dataset) items
    td_cids = sorted({it["cid"] for it in all_items if it.get("src") == "td"})
    random.Random(20260922).shuffle(td_cids)
    hold = set(td_cids[:max(1, int(len(td_cids) * 0.06))]) if td_cids else set()
    is_hold = lambda it: it.get("src") == "td" and it["cid"] in hold
    calib_items = [it for it in all_items if is_hold(it)]
    train_items = [it for it in all_items if not is_hold(it)]
    if not calib_items:   # fallback if items carry no tags
        order = list(range(len(all_items))); random.Random(20260922).shuffle(order)
        n_c = min(400, len(all_items) // 10)
        calib_items = [all_items[i] for i in sorted(order[:n_c])]
        train_items = [all_items[i] for i in sorted(order[n_c:])]
    for it in train_items:
        it["w"] = 1.0 if it.get("src", "td") == "td" else EXTRA_W
    # Trim before sharding so every rank gets the same count (drops at most
    # world_size-1 items). Otherwise ranks can run different numbers of
    # micro-batches and DDP deadlocks on the unmatched all-reduce.
    # See https://github.com/NandhaKishorM/laya/issues/678
    train_items = train_items[:len(train_items) // world_size * world_size]
    my_items = train_items[rank::world_size]
    
    EPOCHS = 6
    MICRO_BATCH = 8      # 8 sequences per forward pass per GPU
    GRAD_ACCUM = 4       # Effective batch across 2 GPUs = 64 sequences (8 * 2 * 4)
    # Experiment 1: lower learning rates; compare against the original V4 baseline.
    LR_ENCODER = 1.5e-5  # Encoder adaptation rate (V4 baseline: 2.5e-5)
    LR_HEAD = 7.5e-5     # Head adaptation rate (V4 baseline: 1.0e-4)
    WARMUP_FRAC = 0.05

    enc_params = [p for n, p in ddp_model.named_parameters() if "encoder." in n]
    head_params = [p for n, p in ddp_model.named_parameters() if "encoder." not in n]
    
    optimizer = torch.optim.AdamW([
        {"params": enc_params, "lr": LR_ENCODER},
        {"params": head_params, "lr": LR_HEAD}
    ], weight_decay=0.01)
    
    updates_per_epoch = math.ceil(len(my_items) / (MICRO_BATCH * GRAD_ACCUM))
    total_updates = max(1, updates_per_epoch * EPOCHS)
    warm = max(1, int(WARMUP_FRAC * total_updates))
    def lr_lambda(step):
        if step < warm:
            return (step + 1) / warm
        prog = min(1.0, (step - warm) / max(1, total_updates - warm))
        return max(0.02, 0.5 * (1 + math.cos(math.pi * prog)))
    scheduler = torch.optim.lr_scheduler.LambdaLR(optimizer, lr_lambda)
    scaler = torch.amp.GradScaler("cuda", enabled=True)
    
    if rank == 0:
        print(f"Starting 2xT4 DDP training: {len(train_items)} train items ({len(calib_items)} held out for calibration) | {len(my_items)} per rank | {EPOCHS} epochs")
    t0 = time.time()
    best = {"nll": float("inf"), "epoch": 0, "sd": None}
    
    for epoch in range(EPOCHS):
        random.seed(42 + epoch + rank)
        random.shuffle(my_items)
        epoch_loss, n_batches = 0.0, 0
        optimizer.zero_grad(set_to_none=True)
        accum_step = 0
        
        for b_idx in range(0, len(my_items), MICRO_BATCH):
            chunk = my_items[b_idx:b_idx + MICRO_BATCH]
            if not chunk:
                continue

            batch = collate_train_batch(chunk, tok.pad_token_id)
            
            with torch.autocast("cuda", dtype=torch.float16):
                logits, act = ddp_model(
                    batch["input_ids"].to(device),
                    batch["attention_mask"].to(device),
                    batch["marker_pos"].to(device),
                    batch["marker_mask"].to(device),
                    batch["qtype"].to(device)
                )
            
            logits = logits.float()
            mask = batch["marker_mask"].to(device)
            k = mask.sum(-1, keepdim=True).float()
            target = batch["target"].to(device)
            
            # Soft cross-entropy on the soft targets only (RL term removed: its reward carried no signal)
            w = batch["weight"].to(device)
            ce_i = -(target * torch.log_softmax(logits.masked_fill(~mask, -1e4), -1)).sum(-1)
            loss_ce = (ce_i * w).sum() / w.sum().clamp_min(1e-6)
            # 0 * sum(params): gives every parameter a (zero) gradient each step, so DDP can run with
            # find_unused_parameters=False even if a micro-batch lacks a question type.
            touch = sum(p.sum() for p in all_params) * 0.0
            loss = loss_ce / GRAD_ACCUM + 0.0 * act.sum() + touch

            scaler.scale(loss).backward()
            accum_step += 1
            
            if accum_step % GRAD_ACCUM == 0 or (b_idx + MICRO_BATCH) >= len(my_items):
                scaler.unscale_(optimizer)
                torch.nn.utils.clip_grad_norm_(ddp_model.parameters(), 1.0)
                scaler.step(optimizer)
                scale_before = scaler.get_scale()
                scaler.update()
                # GradScaler skips optimizer.step() on inf/nan grads (and then lowers the scale);
                # only advance the LR schedule when the optimizer really stepped.
                if scaler.get_scale() >= scale_before:
                    scheduler.step()
                optimizer.zero_grad(set_to_none=True)
            
            epoch_loss += loss.item() * GRAD_ACCUM
            n_batches += 1
            
            if rank == 0 and (n_batches % 50) == 0:
                cur_lr = scheduler.get_last_lr()[0]
                print(f"  Epoch {epoch+1}/{EPOCHS} | Step {n_batches} | Loss: {loss.item()*GRAD_ACCUM:.4f} | LR: {cur_lr:.2e}")

        if rank == 0:
            print(f"=== Epoch {epoch+1}/{EPOCHS} Completed in {time.time()-t0:.1f}s | Avg Loss: {epoch_loss/max(1, n_batches):.4f} ===")

        dist.barrier()

        # Overwrite a single rolling checkpoint after each epoch so a crash,
        # OOM, or Kaggle session timeout doesn't lose all prior training.
        if rank == 0:
            ckpt_dir = os.path.join(output_dir, "checkpoint_latest")
            os.makedirs(ckpt_dir, exist_ok=True)
            ckpt_sd = {k: v.half().contiguous().cpu() for k, v in model.state_dict().items()}
            save_file(ckpt_sd, os.path.join(ckpt_dir, "model.safetensors"))
            model.encoder.config.save_pretrained(os.path.join(ckpt_dir, "encoder"))
            tok.save_pretrained(os.path.join(ckpt_dir, "tokenizer"))
            with open(os.path.join(ckpt_dir, "checkpoint_meta.json"), "w") as f:
                json.dump({
                    "epoch": epoch + 1,
                    "total_epochs": EPOCHS,
                    "avg_loss": epoch_loss / max(1, n_batches)
                }, f, indent=2)
            # Save a valid Laya config alongside every rolling checkpoint.
            ckpt_cfg = dict(cfg)
            ckpt_cfg["fine_tuned"] = True
            ckpt_cfg["model_name"] = "laya-typed-decisions-v4"
            ckpt_cfg.setdefault("temperature", [1.0, 1.0, 1.0])
            ckpt_cfg.setdefault("temperature_by_options", {})
            ckpt_cfg.setdefault("noul_threshold", 0.5)
            ckpt_cfg.setdefault("noul_thresholds", {})
            with open(os.path.join(ckpt_dir, "rl_agent_config.json"), "w") as f:
                json.dump(ckpt_cfg, f, indent=2)
            print(f"  Saved rolling checkpoint (epoch {epoch+1}/{EPOCHS}) to {ckpt_dir}")
            h_nll, h_acc = eval_holdout(model, calib_items, tok, device)
            print(f"  Hold-out soft-CE {h_nll:.4f} | argmax acc {h_acc:.3f}")
            if h_nll < best["nll"]:
                best.update(nll=h_nll, epoch=epoch + 1, sd={k: v.detach().cpu().clone() for k, v in model.state_dict().items()})

    dist.barrier()
    
    # Post-training temperature calibration on rank 0 (micro-batched in chunks of 16 to prevent OOM)
    if rank == 0:
        print("\nFitting post-training calibration temperatures...")
        del optimizer, scaler, scheduler
        torch.cuda.empty_cache()
        if best["sd"] is not None:
            model.load_state_dict(best["sd"]); print(f"Using best epoch {best['epoch']} (hold-out soft-CE {best['nll']:.4f})")
        model.eval()
        # calib_items was held out above and never entered a training batch
        calib_preds = []
        with torch.no_grad():
            for c_idx in range(0, len(calib_items), 16):
                c_chunk = calib_items[c_idx:c_idx + 16]
                cb = collate_train_batch(c_chunk, tok.pad_token_id)
                with torch.autocast("cuda", dtype=torch.float16):
                    l_sub, _ = model(
                        cb["input_ids"].to(device),
                        cb["attention_mask"].to(device),
                        cb["marker_pos"].to(device),
                        cb["marker_mask"].to(device),
                        cb["qtype"].to(device)
                    )
                l_np = l_sub.float().cpu().numpy()
                for r, it in enumerate(c_chunk):
                    k = len(it["markers"])
                    calib_preds.append((it["qtype"], l_np[r, :k], it["target"], it.get("qid", "__all__")))
        
        hacc = np.mean([int(np.argmax(z) == int(np.argmax(t))) for _, z, t, _ in calib_preds])
        print(f"Held-out typed-decisions argmax accuracy: {hacc:.3f} on {len(calib_preds)} items")

        # Per-type scalar (fallback) + per-(qtype, option-count bucket) temperatures.
        # Bucket names come from the runtime's own temp_bucket(), the only keys Agent looks up.
        fitted_temps, tbo = [1.0, 1.0, 1.0], {}
        try:
            by_bucket = {}
            for qt in range(3):
                sel = [(z, t) for q_type, z, t, _ in calib_preds if q_type == qt]
                fitted_temps[qt] = fit_one_temp(sel)
            for q_type, z, t, _ in calib_preds:
                by_bucket.setdefault(temp_bucket(q_type, len(z)), []).append((z, t))
            for qt in range(3):                       # every bucket gets a value, so stale inherited
                for k in (2, 4, 8, 12):               # entries (e.g. a 0.1 sharpener) are overwritten
                    key = temp_bucket(qt, k)
                    pairs = by_bucket.get(key, [])
                    tbo[key] = fit_one_temp(pairs, MIN_BUCKET_N) if len(pairs) >= MIN_BUCKET_N else fitted_temps[qt]
            print("Type temps (choice, score, noul):", [round(t, 3) for t in fitted_temps])
            print("Bucket temps:", {k: round(v, 3) for k, v in tbo.items()}, "| bucket sizes:", {k: len(v) for k, v in by_bucket.items()})
        except Exception as e:
            print("Temperature fitting fallback:", repr(e))

        def T_for(qt, k):
            return tbo.get(temp_bucket(qt, k), fitted_temps[qt])
        confs, corr = [], []
        for qt, z, t, _ in calib_preds:
            if qt == 1:   # score: eval counts exact level; same confidence definition
                pass
            p = softmax_np(z, T_for(qt, len(z)))
            confs.append(p.max()); corr.append(float(np.argmax(p) == int(np.argmax(t))))
        raw_c = [softmax_np(z).max() for _, z, _, _ in calib_preds]
        e_raw, e_cal = ece_np(raw_c, corr), ece_np(confs, corr)
        print(f"Hold-out mean conf (raw) {np.mean(raw_c):.3f} vs top-1 acc {np.mean(corr):.3f}")
        print(f"Hold-out ECE raw {e_raw:.3f} -> bucket-calibrated {e_cal:.3f} (fit set, optimistic)")
        if e_cal > e_raw:
            fitted_temps, tbo = [1.0, 1.0, 1.0], {k: 1.0 for k in tbo}
            print("Calibration was worse than raw on hold-out -> reset all temperatures to 1.0")

        # Noul decision threshold per question id, maximising macro-F1 on the hold-out.
        noul = [(qid, softmax_np(z, T_for(2, len(z)))[1], float(t[1]) >= 0.5) for qt, z, t, qid in calib_preds if qt == 2 and len(z) == 2]
        thr_all = fit_noul_threshold([p for _, p, _ in noul], [y for _, _, y in noul])
        noul_thr = {}
        for qid in sorted({q for q, _, _ in noul}):
            rows = [(p, y) for q, p, y in noul if q == qid]
            thr = fit_noul_threshold([p for p, _ in rows], [y for _, y in rows])
            if qid != "__all__" and thr != 0.5:
                noul_thr[str(qid)] = thr
        print("Noul thresholds disabled (hold-out too small; fitted values such as 0.77 were noise):", noul_thr)
        noul_thr, thr_all = {}, 0.5
        os.makedirs(output_dir, exist_ok=True)
        sd = {k: v.half().contiguous().cpu() for k, v in model.state_dict().items()}
        save_file(sd, os.path.join(output_dir, "model.safetensors"))
        model.encoder.config.save_pretrained(os.path.join(output_dir, "encoder"))
        tok.save_pretrained(os.path.join(output_dir, "tokenizer"))
        
        cfg["fine_tuned"] = True
        cfg["model_name"] = "laya-typed-decisions-v4"
        cfg["temperature"] = fitted_temps
        cfg["temperature_by_options"] = tbo          # kept in config; stale inherited buckets overwritten above
        cfg["noul_threshold"] = thr_all
        cfg["noul_thresholds"] = noul_thr           # read by the eval cell (runtime always reports P(true))
        with open(os.path.join(output_dir, "rl_agent_config.json"), "w") as f:
            json.dump(cfg, f, indent=2)
        print(f"Model successfully saved to {output_dir}!")

    dist.destroy_process_group()

if __name__ == "__main__":
    main()


## 5. Launch Multi-GPU Fine-Tuning with `torchrun`
Runs on both T4 GPUs in parallel (~4 to 6 minutes total).


In [ ]:
import os, subprocess, sys

OUTPUT_DIR = "/kaggle/working/laya_finetuned_typed_decisions"

# `model_dir` is created by Section 3 from the actual Hugging Face download.
if "model_dir" not in globals():
    raise RuntimeError(
        "Base model path `model_dir` is missing. Run Section 3 successfully first; "
        "do not replace this with a guessed folder path."
    )
MODEL_DIR = os.path.abspath(os.fspath(model_dir))
if not os.path.isdir(MODEL_DIR):
    raise FileNotFoundError(f"Downloaded base-model folder is missing: {MODEL_DIR}")
if not os.path.isfile("/kaggle/working/train_ddp.py"):
    raise FileNotFoundError("Training script is missing. Run Section 4 to create train_ddp.py.")

os.makedirs(OUTPUT_DIR, exist_ok=True)
cmd = [sys.executable, "-m", "torch.distributed.run", "--standalone", "--nproc_per_node=2",
       "/kaggle/working/train_ddp.py", MODEL_DIR, OUTPUT_DIR]
print("Executing DDP training:", " ".join(cmd))
result = subprocess.run(cmd, check=False)
if result.returncode != 0:
    raise RuntimeError(
        f"DDP training failed with exit code {result.returncode}. "
        "Do not run Section 6 until a checkpoint is available; inspect the training traceback above."
    )
print("Training process exited successfully. Section 6 will verify the checkpoint before evaluation.")

## 6. Run Benchmark Evaluation on Test Split (400 cases / 2,000 decisions)
Evaluate the fine-tuned model against the official `test` split.


**Checkpoint recovery:** Section 6 can recover the latest saved epoch if training is interrupted. Any metrics from a recovered checkpoint must be labeled as a partial run; for final results, allow all six epochs and calibration to complete.

In [ ]:
import os, time, json, shutil
import numpy as np
import pandas as pd
from datasets import load_dataset
import laya
from laya.common import ece_score

OUTPUT_DIR = "/kaggle/working/laya_finetuned_typed_decisions"
if "model_dir" not in globals() or not os.path.isdir(str(model_dir)):
    raise RuntimeError("Base model path is missing. Run Section 3 successfully before evaluation.")

# Recover the latest saved checkpoint if the final model has not been written yet.
_ckpt_dir = os.path.join(OUTPUT_DIR, "checkpoint_latest")
_root_model = os.path.join(OUTPUT_DIR, "model.safetensors")
_ckpt_model = os.path.join(_ckpt_dir, "model.safetensors")
if not os.path.isfile(_root_model) and os.path.isfile(_ckpt_model):
    os.makedirs(OUTPUT_DIR, exist_ok=True)
    shutil.copy2(_ckpt_model, _root_model)
    for _folder in ("encoder", "tokenizer"):
        _src = os.path.join(_ckpt_dir, _folder)
        _dst = os.path.join(OUTPUT_DIR, _folder)
        if os.path.isdir(_src):
            if os.path.exists(_dst): shutil.rmtree(_dst)
            shutil.copytree(_src, _dst)
    _root_cfg = os.path.join(OUTPUT_DIR, "rl_agent_config.json")
    if not os.path.isfile(_root_cfg):
        if "cfg" not in globals():
            raise RuntimeError("Partial checkpoint found, but base config `cfg` is missing. Run Section 3.")
        _base_cfg = dict(cfg)
        _base_cfg.update({"fine_tuned": True, "model_name": "laya-typed-decisions-partial-checkpoint"})
        _base_cfg.setdefault("temperature", [1.0, 1.0, 1.0])
        _base_cfg.setdefault("temperature_by_options", {})
        _base_cfg.setdefault("noul_threshold", 0.5)
        _base_cfg.setdefault("noul_thresholds", {})
        with open(_root_cfg, "w", encoding="utf-8") as f:
            json.dump(_base_cfg, f, indent=2)
    print("WARNING: evaluating a recovered PARTIAL checkpoint, not a completed training run.")

if not os.path.isfile(_root_model):
    raise FileNotFoundError(
        f"No trained checkpoint found at {_root_model} or {_ckpt_model}. "
        "Section 6 cannot evaluate the base model as if it were fine-tuned. Complete Section 5 first."
    )
if not os.path.isfile(os.path.join(OUTPUT_DIR, "rl_agent_config.json")):
    raise FileNotFoundError("Checkpoint config is missing: rl_agent_config.json")

print("Loading official test split for evaluation...")
ds_test = load_dataset("LocalLLaMA/typed-decisions", "all", split="test")
agent_ft = laya.Agent(OUTPUT_DIR, device="cuda")

predictions = []
latencies_ms = []
print(f"Evaluating {len(ds_test)} test cases on GPU...")
t0_eval = time.time()
for i, row in enumerate(ds_test):
    case_id = row["id"]
    workflow = row["workflow"]
    state = json.loads(row["state"])
    questions = json.loads(row["questions"])
    gold = json.loads(row["gold"])
    t0 = time.perf_counter()
    res = agent_ft.predict(state, questions)
    dt_ms = (time.perf_counter() - t0) * 1000
    latencies_ms.append(dt_ms)
    predictions.append({"id": case_id, "workflow": workflow, "pred": res["answers"],
                        "gold": gold, "questions": questions, "latency_ms": dt_ms})
print(f"Evaluated all {len(ds_test)} cases in {time.time() - t0_eval:.1f}s!")

## 7. Compute Official Metrics & Comparison Table against TypeSafe Jev


In [ ]:
import os, json
_cfg = json.load(open(os.path.join(OUTPUT_DIR, "rl_agent_config.json")))
NOUL_DEF, NOUL_THR = _cfg.get("noul_threshold", 0.5), _cfg.get("noul_thresholds", {})
accuracies = []
soft_accuracies = []
brier_scores = []
kl_divs = []
tv_distances = []
score_maes = []
within_one = []
all_confs = []
all_corrects = []

for item in predictions:
    pred_answers = item["pred"]
    gold_answers = item["gold"]
    questions = item["questions"]
    
    for qid, qdef in questions.items():
        p_ans = pred_answers[qid]
        g_ans = gold_answers[qid]
        q_type = qdef["type"]
        
        # 1. Choice
        if q_type == "choice":
            keys = list(qdef["criteria"].keys())
            pred_choice = p_ans["choice"]
            gold_label = str(g_ans["label"])
            
            is_corr = float(pred_choice == gold_label)
            accuracies.append(is_corr)
            all_corrects.append(is_corr)
            
            p_probs = np.array([p_ans["probabilities"].get(k, 1e-6) for k in keys])
            g_probs = np.array([g_ans["probabilities"].get(k, 1e-6) for k in keys])
            p_probs /= p_probs.sum()
            g_probs /= g_probs.sum()
            
            all_confs.append(float(p_probs.max()))
            soft_accuracies.append(float((p_probs * g_probs).sum()))
            brier_scores.append(float(((p_probs - g_probs) ** 2).sum()))
            tv_distances.append(float(0.5 * np.abs(p_probs - g_probs).sum()))
            kl_divs.append(float((g_probs * np.log(np.clip(g_probs / p_probs, 1e-12, 1e4))).sum()))
            
        # 2. Noul
        elif q_type == "noul":
            p_val = p_ans["noul"]
            g_val = g_ans.get("noul", g_ans.get("probabilities", {}).get("true", 0.5))
            gold_label = str(g_ans["label"]).lower()
            
            pred_label = "true" if p_val >= NOUL_THR.get(qid, NOUL_DEF) else "false"
            is_corr = float(pred_label == gold_label)
            accuracies.append(is_corr)
            all_corrects.append(is_corr)
            all_confs.append(float(max(p_val, 1.0 - p_val)))
            
            p_dist = np.array([1.0 - p_val, p_val])
            g_dist = np.array([1.0 - g_val, g_val])
            
            soft_accuracies.append(float((p_dist * g_dist).sum()))
            brier_scores.append(float(((p_dist - g_dist) ** 2).sum()))
            tv_distances.append(float(0.5 * np.abs(p_dist - g_dist).sum()))
            kl_divs.append(float((g_dist * np.log(np.clip(g_dist / p_dist, 1e-12, 1e4))).sum()))
            
        # 3. Score
        elif q_type == "score":
            p_score = p_ans["score"]
            g_score = g_ans.get("score", 0.0)
            score_maes.append(abs(p_score - g_score))
            within_one.append(float(abs(p_score - g_score) <= 1.0))
            
            # Use argmax of probabilities for discrete prediction
            n_levels = len(qdef.get("criteria", []))
            p_probs_score = np.array([p_ans["probabilities"].get(str(i), 0.0) for i in range(n_levels)])
            if p_probs_score.sum() > 0:
                p_probs_score /= p_probs_score.sum()
                p_lvl = int(np.argmax(p_probs_score))
                all_confs.append(float(p_probs_score.max()))
            else:
                p_lvl = int(round(p_score))
                all_confs.append(0.5)
                
            g_lvl = int(g_ans.get("label", int(round(g_score))))
            is_corr = float(p_lvl == g_lvl)
            accuracies.append(is_corr)
            all_corrects.append(is_corr)

laya_acc = np.mean(accuracies)
laya_soft_acc = np.mean(soft_accuracies)
laya_brier = np.mean(brier_scores)
laya_kl = np.mean(kl_divs)
laya_tv = np.mean(tv_distances)
laya_ece = ece_score(np.array(all_confs), np.array(all_corrects))
laya_mae = np.mean(score_maes) if score_maes else 0.0
laya_within1 = np.mean(within_one) if within_one else 0.0
laya_latency = np.percentile(latencies_ms, 50)

comparison_data = [
    {
        "Model": "TypeSafe Jev 1.13.0",
        "Kind": "general",
        "Accuracy": 0.727,
        "Soft Acc": 0.580,
        "Brier": 0.148,
        "ECE": 0.144,
        "Score MAE": 0.391,
        "Within 1 Level": "0.952",
        "ms/case": 710,
        "Cost/Case": "$0.0004 (API)"
    },
    {
        "Model": "Laya V2 (Fine-Tuned 2xT4)",
        "Kind": "fine-tuned",
        "Accuracy": round(laya_acc, 3),
        "Soft Acc": round(laya_soft_acc, 3),
        "Brier": round(laya_brier, 3),
        "ECE": round(laya_ece, 3),
        "Score MAE": round(laya_mae, 3),
        "Within 1 Level": f"{laya_within1:.3f}",
        "ms/case": round(laya_latency, 1),
        "Cost/Case": "$0.00 (Self-Hosted)"
    },
    {
        "Model": "Laya V1 (previous run)",
        "Kind": "fine-tuned",
        "Accuracy": 0.764, "Soft Acc": 0.534, "Brier": 0.059, "ECE": 0.223, "Score MAE": 0.317,
        "Within 1 Level": "0.990", "ms/case": 143.4, "Cost/Case": "$0.00 (Self-Hosted)"
    },
    {
        "Model": "ModernBERT-base (149M)",
        "Kind": "specialist",
        "Accuracy": 0.646,
        "Soft Acc": 0.542,
        "Brier": 0.119,
        "ECE": 0.179,
        "Score MAE": 0.444,
        "Within 1 Level": "0.931",
        "ms/case": 349,
        "Cost/Case": "$0.00"
    },
    {
        "Model": "Teacher Self-Agreement",
        "Kind": "ceiling",
        "Accuracy": 0.735,
        "Soft Acc": "-",
        "Brier": "-",
        "ECE": "-",
        "Score MAE": "-",
        "Within 1 Level": "-",
        "ms/case": "-",
        "Cost/Case": "-"
    }
]

df_comp = pd.DataFrame(comparison_data)
print("=== HEAD-TO-HEAD BENCHMARK TABLE ===\n")
print(df_comp.to_markdown(index=False))


## 8. (Optional) Push to Hugging Face Hub


In [ ]:
import os
from huggingface_hub import HfApi
# Make sure you have added HF_TOKEN to Kaggle Secrets (Add-ons -> Secrets -> HF_TOKEN)
from kaggle_secrets import UserSecretsClient
from huggingface_hub import login

secrets = UserSecretsClient()

token = secrets.get_secret("HF_TOKEN")

login(token=token)

print("Hugging Face login successful.")

# The token can only write to its own account (or orgs it belongs to), so publish under the token owner's namespace.
# If your token is a fine-grained one, it must have "Write access to contents/settings of all repos under your personal namespace".
_user = HfApi(token=token).whoami()["name"]
NEW_REPO = f"{_user}/laya-typed-decisions-v2"
print(f"Publishing {laya_acc:.3f} accuracy fine-tuned model to {NEW_REPO}...")

# Reference baselines pulled from the comparison table above, so the model
# card's narrative always matches the numbers actually computed this run.
JEV_ACC = next(r["Accuracy"] for r in comparison_data if r["Model"] == "TypeSafe Jev 1.13.0")
CEILING_ACC = next(r["Accuracy"] for r in comparison_data if r["Model"] == "Teacher Self-Agreement")

def describe_vs_baseline(acc, jev_acc=JEV_ACC, ceiling_acc=CEILING_ACC):
    """Builds an accuracy comparison sentence that reflects this run's actual result,
    instead of assuming the fine-tuned model always wins."""
    if acc > ceiling_acc:
        return (f"outperforming **TypeSafe Jev 1.13.0 ({jev_acc:.3f})** and surpassing the "
                f"benchmark's **Teacher Self-Agreement ceiling ({ceiling_acc:.3f})**")
    elif acc > jev_acc:
        return (f"outperforming **TypeSafe Jev 1.13.0 ({jev_acc:.3f})**, though still below the "
                f"benchmark's Teacher Self-Agreement ceiling ({ceiling_acc:.3f})")
    else:
        return (f"trailing **TypeSafe Jev 1.13.0 ({jev_acc:.3f})** and the benchmark's "
                f"Teacher Self-Agreement ceiling ({ceiling_acc:.3f})")

comparison_sentence = describe_vs_baseline(laya_acc)

# 1. Write comprehensive model card README.md into OUTPUT_DIR before upload
readme_content = f"""---
license: apache-2.0
library_name: transformers
tags:
- laya
- system-one
- calibrated-decisions
- rlcd
- structured-decisions
- typed-decisions
- benchmark
metrics:
- accuracy
- brier_score
model-index:
- name: laya-typed-decisions
  results:
  - task:
      type: text-classification
      name: System One Decision Benchmark
    dataset:
      type: LocalLLaMA/typed-decisions
      name: Typed Decisions
    metrics:
    - type: accuracy
      value: {laya_acc:.3f}
    - type: brier_score
      value: {laya_brier:.3f}
---

# Laya (Fine-Tuned on Typed-Decisions Benchmark)

This is **Laya** fine-tuned on the 1,200 training cases (6,000 decisions) of the independent [LocalLLaMA/typed-decisions](https://huggingface.co/datasets/LocalLLaMA/typed-decisions) benchmark.

On the official 400-case test set (2,000 decisions across Agent Trace Observability, Customer Service, Invoice Processing, and Security Incidents), it achieves **{laya_acc:.3f} Accuracy**, {comparison_sentence}.

## Head-to-Head Benchmark Results

| Model | Kind | Accuracy | Soft Acc | Brier Score | ECE | Score MAE | Within 1 Level | Latency (p50) | Cost/Case |
|---|---|---|---|---|---|---|---|---|---|
| **Laya (Ours)** | **fine-tuned** | **{laya_acc:.3f}** | **{laya_soft_acc:.3f}** | **{laya_brier:.3f}** | **{laya_ece:.3f}** | **{laya_mae:.3f}** | **{laya_within1:.3f}** | **{laya_latency:.1f} ms** | **$0.00 (Self-Hosted)** |
| TypeSafe Jev 1.13.0 | general | 0.727 | 0.580 | 0.148 | 0.144 | 0.391 | 0.952 | 710 ms | $0.0004 (API) |
| ModernBERT-base (149M) | specialist | 0.646 | 0.542 | 0.119 | 0.179 | 0.444 | 0.931 | 349 ms | $0.00 |
| Teacher Self-Agreement | ceiling | 0.735 | - | - | - | - | - | - | - |

## Installation & Quickstart

```bash
pip install laya
```

```python
import laya

# Load the fine-tuned model directly from Hugging Face
agent = laya.load("{NEW_REPO}")

# Evaluate any workflow state and typed questions in a single forward pass
result = agent.predict(state, questions)
print(result["answers"])
```

## License
Apache 2.0. Developed by [Convai Innovations](https://huggingface.co/convaiinnovations).
"""

with open(os.path.join(OUTPUT_DIR, "README.md"), "w") as f:
    f.write(readme_content)

print(f"Generated model card at {os.path.join(OUTPUT_DIR, 'README.md')}")

api = HfApi(token=token)
api.create_repo(NEW_REPO, repo_type="model", private=False, exist_ok=True)

# Upload the fine-tuned weights, config, tokenizer, and README
api.upload_folder(
    folder_path=OUTPUT_DIR,
    repo_id=NEW_REPO,
    repo_type="model",
    commit_message=f"Laya fine-tuned on typed-decisions: Accuracy {laya_acc:.3f} vs Jev {JEV_ACC:.3f} / Ceiling {CEILING_ACC:.3f}"
)

print(f"\nModel successfully published to: https://huggingface.co/{NEW_REPO}")


## 9. Save Benchmark Report as JSON
Export the full evaluation metrics, comparison table, and per-workflow accuracy breakdown to JSON.


In [ ]:
import json, os

report = {
    "benchmark": "LocalLLaMA/typed-decisions",
    "model": "Laya V2 (Fine-Tuned 2xT4)",
    "n_cases": len(ds_test),
    "n_decisions": len(ds_test) * 5,
    "metrics": {
        "accuracy": round(float(laya_acc), 4),
        "soft_accuracy": round(float(laya_soft_acc), 4),
        "brier_score": round(float(laya_brier), 4),
        "ece": round(float(laya_ece), 4),
        "score_mae": round(float(laya_mae), 4),
        "within_1_level": round(float(laya_within1), 4),
        "latency_p50_ms": round(float(laya_latency), 1),
        "latency_p95_ms": round(float(np.percentile(latencies_ms, 95)), 1),
        "kl_divergence": round(float(laya_kl), 4),
        "total_variation": round(float(laya_tv), 4)
    },
    "comparison": comparison_data,
    "per_workflow": {}
}

# Calculate per-workflow accuracy
for wf in ["agent_trace_observability", "customer_service", "invoice_processing", "security_incidents"]:
    wf_items = [item for item in predictions if item["workflow"] == wf]
    wf_corr = []
    for item in wf_items:
        for qid, qdef in item["questions"].items():
            p_ans = item["pred"][qid]
            g_ans = item["gold"][qid]
            if qdef["type"] == "choice":
                wf_corr.append(float(p_ans["choice"] == str(g_ans["label"])))
            elif qdef["type"] == "noul":
                pred_label = "true" if p_ans["noul"] >= NOUL_THR.get(qid, NOUL_DEF) else "false"
                wf_corr.append(float(pred_label == str(g_ans["label"]).lower()))
            elif qdef["type"] == "score":
                p_probs = [p_ans["probabilities"].get(str(i), 0.0) for i in range(len(qdef.get("criteria", [])))]
                p_lvl = int(np.argmax(p_probs)) if sum(p_probs) > 0 else int(round(p_ans["score"]))
                g_lvl = int(g_ans.get("label", int(round(g_ans.get("score", 0.0)))))
                wf_corr.append(float(p_lvl == g_lvl))
    if wf_corr:
        report["per_workflow"][wf] = {
            "n_decisions": len(wf_corr),
            "accuracy": round(float(np.mean(wf_corr)), 4)
        }

# Save in working directory and model directory
out_file1 = "/kaggle/working/laya_typed_decisions_benchmark_report.json"
out_file2 = os.path.join(OUTPUT_DIR, "benchmark_report.json")

with open(out_file1, "w") as f:
    json.dump(report, f, indent=2)

with open(out_file2, "w") as f:
    json.dump(report, f, indent=2)

print(f"Benchmark results successfully saved to:")
print(f"  - {out_file1}")
print(f"  - {out_file2}")
print("\nSummary:")
print(json.dumps(report["metrics"], indent=2))
print("\nPer-Workflow Accuracy:")
print(json.dumps(report["per_workflow"], indent=2))


In [ ]:
import shutil

OUTPUT_DIR = "/kaggle/working/laya_finetuned_typed_decisions"

shutil.make_archive(
    "/kaggle/working/laya_finetuned_typed_decisions",
    "zip",
    OUTPUT_DIR
)

print("ZIP created successfully.")

In [ ]:
import os

OUTPUT_DIR = "/kaggle/working/laya_finetuned_typed_decisions"

print("Output folder exists:", os.path.exists(OUTPUT_DIR))
print("Checkpoint exists:", os.path.exists(
    os.path.join(OUTPUT_DIR, "checkpoint_latest", "model.safetensors")
))
print("Config exists:", os.path.exists(
    os.path.join(OUTPUT_DIR, "rl_agent_config.json")
))

In [ ]:
from IPython.display import FileLink, display

display(FileLink(
    "/kaggle/working/laya_typed_decisions_benchmark_report.json"
))

In [ ]:
import os
from IPython.display import FileLink, display

path = "/kaggle/working/laya_typed_decisions_benchmark_report.json"

print("Report exists:", os.path.exists(path))

if os.path.exists(path):
    display(FileLink(path))
else:
    print("Report not found in this session.")

In [ ]:
import os
import zipfile
from IPython.display import FileLink, display

OUTPUT_DIR = "/kaggle/working/laya_finetuned_typed_decisions"
REPORT = "/kaggle/working/laya_typed_decisions_benchmark_report.json"
ZIP_PATH = "/kaggle/working/benchmark_report.zip"

# Find the report in either saved location
if not os.path.exists(REPORT):
    REPORT = os.path.join(OUTPUT_DIR, "benchmark_report.json")

if os.path.exists(REPORT):
    with zipfile.ZipFile(ZIP_PATH, "w", zipfile.ZIP_DEFLATED) as z:
        z.write(REPORT, arcname="benchmark_report.json")

    print("Benchmark report packaged successfully!")
    display(FileLink(ZIP_PATH))
else:
    print("Report not found. Files currently available:")
    print(os.listdir("/kaggle/working"))